# Bradley-Terry Reward Modeling

*Reference list: Christiano et al. 2017, Ziegler et al. 2019.*

Preference learning from pairwise comparisons. Full derivation of the Bradley-Terry model, training, and Goodhart's law demonstration.

In [ ]:
import sys, os
from pathlib import Path
_p = Path.cwd()
while _p != _p.parent and not (_p / 'utils' / 'reward_model.py').exists():
    _p = _p.parent
REPO = str(_p)
sys.path.insert(0, REPO)

from utils.reward_model import BradleyTerry, PreferenceDataset, train_rm, make_synthetic_preferences
from utils.env import ENV
import torch, numpy as np, matplotlib.pyplot as plt
ENV.summary()

## 1. The Bradley-Terry Model

Given two responses $y_w$ (winner) and $y_l$ (loser) to prompt $x$, the Bradley-Terry model assigns probability:

$$P[y_w \succ y_l \mid x] = \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))$$

where $\sigma$ is the sigmoid and $r_\phi$ is the learned reward model.

The **log-likelihood** of a preference dataset $\mathcal{D} = \{(x,y_w,y_l)\}$:

$$\mathcal{L}(\phi) = \mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}\!\left[\log \sigma(r_\phi(x,y_w) - r_\phi(x,y_l))\right]$$

This is binary cross-entropy on reward differences — the same loss as logistic regression on the margin $r_w - r_l$.

In [ ]:
# Verify: BT loss = BCE on reward difference
r_w = torch.tensor([2.0, 1.0, 0.5])
r_l = torch.tensor([0.0, 1.5, 1.0])
bt_loss  = -torch.nn.functional.logsigmoid(r_w - r_l).mean()
bce_loss = torch.nn.functional.binary_cross_entropy(
    torch.sigmoid(r_w - r_l), torch.ones(3)
)
print(f'BT loss:  {bt_loss:.6f}')
print(f'BCE loss: {bce_loss:.6f}')
print(f'Equal: {torch.isclose(bt_loss, bce_loss)}')

## 2. Synthetic Preference Data

Ground truth: $r^*(x) = w^* \cdot x$, $w^* \sim \mathcal{N}(0,I)$ normalised.  
Preference: $y_w \succ y_l$ iff $r^*(y_w) > r^*(y_l)$, with label noise $p_{\text{flip}}$.  

We check whether training recovers $w^*$ via cosine similarity.

In [ ]:
d = 16  # feature dimension
dataset, w_star = make_synthetic_preferences(d=d, n=1024, noise=0.1, seed=0)
print(f'Dataset size: {len(dataset)} pairs')
print(f'w*: {w_star.round(3)}')

In [ ]:
model = BradleyTerry(d_input=d)
history = train_rm(model, dataset, n_epochs=200, lr=1e-2, batch_size=64)

w_learned = model.head.weight.data.squeeze().numpy()
w_learned /= np.linalg.norm(w_learned)
cos_sim = float(np.dot(w_learned, w_star))

plt.figure(figsize=(6,3))
plt.plot(history['loss'])
plt.xlabel('epoch'); plt.ylabel('BT loss')
plt.title(f'Reward model training — final cos(w_learned, w*) = {cos_sim:.4f}')
plt.tight_layout(); plt.show()

## 3. Sample Efficiency: Recovery vs Dataset Size

How much preference data is needed to recover $w^*$?

In [ ]:
Ns = [32, 64, 128, 256, 512, 1024, 2048]
cos_sims = []
for n in Ns:
    ds, w_star = make_synthetic_preferences(d=16, n=n, noise=0.1, seed=0)
    m = BradleyTerry(d_input=16)
    train_rm(m, ds, n_epochs=300, lr=1e-2, batch_size=min(64,n))
    w_l = m.head.weight.data.squeeze().numpy()
    w_l /= np.linalg.norm(w_l)
    cos_sims.append(float(np.dot(w_l, w_star)))

plt.figure(figsize=(6,3))
plt.semilogx(Ns, cos_sims, 'o-')
plt.xlabel('n preference pairs'); plt.ylabel('cos(w_learned, w*)')
plt.title('Reward model recovery vs dataset size')
plt.axhline(1.0, color='gray', ls='--')
plt.tight_layout(); plt.show()

## 4. Goodhart's Law: Optimising Against the RM

Goodhart needs an **exploitable gap** between the proxy $\hat{r}_\phi$ and the gold reward $r^*$. A well-specified linear RM that recovers $w^*$ has no such gap — and label noise doesn't create one, it only lowers the recovery ceiling (the proxy stays an *unbiased* estimate of $w^*$). So naive "optimise $\hat r$, watch $r^*$" on the linear setup shows no overoptimisation.

We create a real gap by making the gold reward trustworthy **only on the data manifold**:

$$r^*(x) = w^* \cdot x \; - \; \lambda \lVert x \rVert^2 \qquad (\text{rewards direction, penalises norm})$$

Training pairs all live on the unit sphere, so $\lVert x \rVert$ is identical within every pair and the penalty **cancels** in the Bradley-Terry comparison — the linear RM still recovers $w^*$ cleanly (it never sees the penalty). When we then gradient-ascend $\hat r_\phi$ on raw $x$, the norm grows, we walk **off the manifold**, and the linear proxy keeps climbing while gold peaks and collapses. *That* is overoptimisation: the proxy is valid in-distribution and optimisation pushes you out of it.

In [ ]:
torch.manual_seed(0)
rng = np.random.default_rng(0)
d, LAM = 16, 0.05
w_star = rng.standard_normal(d); w_star /= np.linalg.norm(w_star)
w_star_t = torch.tensor(w_star, dtype=torch.float32)

def gold(x):                       # gold reward — only trustworthy near the unit sphere
    return x @ w_star_t - LAM * (x ** 2).sum(-1)

def on_sphere(n):                  # the data manifold: unit-norm feature vectors
    z = rng.standard_normal((n, d))
    return z / np.linalg.norm(z, axis=1, keepdims=True)

# Preferences from gold; equal norm within each pair => penalty cancels => ordering is linear
pairs = []
for _ in range(2048):
    a, b = on_sphere(1)[0], on_sphere(1)[0]
    flip = rng.random() < 0.1
    pairs.append((a, b) if ((w_star @ a > w_star @ b) ^ flip) else (b, a))

rm = BradleyTerry(d_input=d)
train_rm(rm, PreferenceDataset(pairs), n_epochs=300, lr=1e-2)
w_l = rm.head.weight.data.squeeze().numpy(); w_l /= np.linalg.norm(w_l)
print(f"RM still recovers w* despite the noise: cos = {float(w_l @ w_star):.4f}")

# Gradient-ascend raw x against the RM — NO normalisation, so ||x|| is free to grow off-manifold
x = torch.randn(64, d, requires_grad=True)
opt_x = torch.optim.Adam([x], lr=0.02)
steps, rm_scores, true_scores, norms = [], [], [], []
for step in range(300):
    loss = -rm(x).mean()
    opt_x.zero_grad(); loss.backward(); opt_x.step()
    if step % 15 == 0:
        with torch.no_grad():
            steps.append(step)
            rm_scores.append(rm(x).mean().item())
            true_scores.append(gold(x).mean().item())
            norms.append(x.norm(dim=-1).mean().item())

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 3.8))
axL.plot(steps, rm_scores, 'b-o', ms=4, label=r'proxy $\hat{r}_\phi$')
axLt = axL.twinx()
axLt.plot(steps, true_scores, 'r-s', ms=4, label=r'gold $r^*$')
axL.set_xlabel('optimisation steps'); axL.set_ylabel('proxy reward', color='b')
axLt.set_ylabel('gold reward', color='r')
axL.set_title("Proxy climbs monotonically, gold peaks then collapses")
axL.legend(loc='upper left'); axLt.legend(loc='lower left')

axR.plot(norms, true_scores, 'r-s', ms=4)
axR.axvline(1.0, color='gray', ls='--', label=r'training manifold $\|x\|=1$')
axR.set_xlabel(r'$\|x\|$ (distance off-manifold)'); axR.set_ylabel('gold reward')
axR.set_title('Gold collapses as optimisation leaves the manifold')
axR.legend(loc='upper right')
plt.tight_layout(); plt.show()